In [1]:
import numpy as np
import gamspy as gams
import pandas as pd
import sys

# For detailed instructions from GAMS, see:
# https://gamspy.readthedocs.io/en/latest/user/notebooks/trnsport.html


def print_records(vars):
    for k in np.arange(0, len(vars)):
        print(vars[k].name, ':')
        display(vars[k].records)
        print()


m = gams.Container()

i = gams.Set(container=m,
             name='i',
             description='Plant')

j = gams.Set(container=m,
             name='j',
             description='Market')

i.setRecords(['Seattle', 'San Diego'])
j.setRecords(['New York', 'Chicago', 'Topeka'])

print_records([i, j])

a = gams.Parameter(container=m,
                   name='a',
                   domain=i,
                   description='Supply of commodity at plant i')

b = gams.Parameter(container=m,
                   name='b',
                   domain=j,
                   description='Demand for commodity at market j')

a.setRecords([('Seattle', 350), ('San Diego', 600)])
b.setRecords([('New York', 325), ('Chicago', 300), ('Topeka', 275)])

print_records([a, b])

distances = pd.DataFrame(
    [
        ['Seattle', 'New York', 2.5],
        ['Seattle', 'Chicago', 1.7],
        ['Seattle', 'Topeka', 1.8],
        ['San Diego', 'New York', 2.5],
        ['San Diego', 'Chicago', 1.8],
        ['San Diego', 'Topeka', 1.4],
    ],
    columns=['From', 'To', 'Distance']
).set_index(['From', 'To'])

d = gams.Parameter(container=m,
                   name='d',
                   domain=[i, j],
                   description='Distance between plant i and market j',
                   records=distances.reset_index(),)

c = gams.Parameter(container=m,
                   name='c',
                   domain=[i, j],
                   description='Cost per unit of shipment between' +
                   'plant i and market j')

x = gams.Variable(container=m,
                  name='x',
                  domain=[i, j],
                  type='Positive',
                  description='Amount of commodity to ship from' +
                  'plant i to market j')

freight_cost = 90
cost = freight_cost * distances / 1000
c.setRecords(cost.reset_index())
c[i, j] = freight_cost*d[i, j] / 1000

print_records([d, c])

supply = gams.Equation(container=m,
                       name='supply',
                       domain=i,
                       description='Observe supply limit at plant i')

demand = gams.Equation(container=m,
                       name='demand',
                       domain=j,
                       description='Satisfy demand at market j')

supply[i] = gams.Sum(j, x[i, j]) <= a[i]
demand[j] = gams.Sum(i, x[i, j]) >= b[j]

obj = gams.Sum((i, j), c[i, j] * x[i, j])  # objective function

transport = gams.Model(container=m,
                       name='transport',
                       equations=[supply, demand],
                       problem='LP',
                       sense=gams.Sense.MIN,
                       objective=obj)  # problem

transport.solve(output=sys.stdout,
                options=gams.Options(equation_listing_limit=10,
                                     variable_listing_limit=10))

print(transport.getEquationListing())
print()
print(transport.getVariableListing())
print()
print(supply.getEquationListing())
print()
print(demand.getEquationListing())
print()
print(x.getVariableListing())
print()
print_records([x])
print('Optimum: ', transport.objective_value)

i :


,uni,element_text
0,Seattle,
1,San Diego,



j :


,uni,element_text
0,New York,
1,Chicago,
2,Topeka,



a :


...


c :


,i,j,value
0,Seattle,New York,0.225
1,Seattle,Chicago,0.153
2,Seattle,Topeka,0.162
3,San Diego,New York,0.225
4,San Diego,Chicago,0.162
5,San Diego,Topeka,0.126



--- Job _SO5v4EldTZ_xZBIZMobBQg.gms Start 09/29/26 15:18:52 50.2.0 d60bb663 WEX-WEI x86 64bit/MS Windows
--- Applying:
    C:\Users\gajam\AppData\Local\Programs\Python\Python313\Lib\site-packages\gamspy_base\gmsprmNT.txt
--- GAMS Parameters defined
    LP cplex
    Input C:\Users\gajam\AppData\Local\Temp\tmpc2drioyt\_SO5v4EldTZ_xZBIZMobBQg.gms
    Output C:\Users\gajam\AppData\Local\Temp\tmpc2drioyt\_SO5v4EldTZ_xZBIZMobBQg.lst
    ScrDir C:\Users\gajam\AppData\Local\Temp\tmpc2drioyt\tmpmq85n538\
    SysDir C:\Users\gajam\AppData\Local\Programs\Python\Python313\Lib\site-packages\gamspy_base\
    LogOption 3
    Trace C:\Users\gajam\AppData\Local\Temp\tmpc2drioyt\_SO5v4EldTZ_xZBIZMobBQg.txt
    License C:\Users\gajam\Documents\GAMSPy\gamspy_license.txt
    OptFile 0
    OptDir C:\Users\gajam\AppData\Local\Temp\tmpc2drioyt\
    LimRow 10
    LimCol 10
    TraceOpt 3
    GDX C:\Users\gajam\AppData\Local\Temp\tmpc2drioyt\_SO5v4EldTZ_xZBIZMobBQgout.gdx
    SolPrint 0
    PreviousWork 1
    

,i,j,level,marginal,lower,upper,scale
0,Seattle,New York,50.0,0.000,0.0,inf,1.0
1,Seattle,Chicago,300.0,0.000,0.0,inf,1.0
2,Seattle,Topeka,0.0,0.036,0.0,inf,1.0
3,San Diego,New York,275.0,0.000,0.0,inf,1.0
4,San Diego,Chicago,0.0,0.009,0.0,inf,1.0
5,San Diego,Topeka,275.0,0.000,0.0,inf,1.0



Optimum:  153.675
